## Download Model from HuggingFace and Upload to S3

This notebook downloads the Qwen3-4B-Instruct-2507 model artifacts from HuggingFace to a local `models/` folder, then uploads them to S3. Both operations are idempotent — re-running this notebook will skip already-downloaded and already-uploaded files.

Run this notebook **once** before running `05.00_fmops_examples.ipynb` or `05.01_fine-tuning-pipeline.ipynb`.

### 1. Install Dependencies

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade --quiet

In [ ]:
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)

### 2. Configuration

In [ ]:
from sagemaker.core.helper.session_helper import Session

sagemaker_session = Session()
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

model_id = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Bucket: {bucket_name}")
print(f"Prefix: {default_prefix}")
print(f"Model: {model_id}")

### 3. Download from HuggingFace and Upload to S3

This uses the shared `ensure_model_on_s3()` utility which:
1. Downloads model files from HuggingFace to `./models/` (skips existing files)
2. Uploads all files to S3 (skips files already in S3)
3. Returns the S3 URI for use by other notebooks

In [ ]:
from steps.model_utils import ensure_model_on_s3

model_s3_destination = ensure_model_on_s3(
    model_id=model_id,
    bucket_name=bucket_name,
    default_prefix=default_prefix,
)

print(f"\nModel S3 location: {model_s3_destination}")
print("\nDone! You can now run 05.00 or 05.01 notebooks.")

### 4. Verify Upload (Optional)

List files in S3 to confirm the model was uploaded correctly.

In [ ]:
import boto3

s3_client = boto3.client('s3')
model_id_filesafe = model_id.replace('/', '_').replace('.', '_')

if default_prefix:
    prefix = f"{default_prefix}/models/{model_id_filesafe}"
else:
    prefix = f"models/{model_id_filesafe}"

response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

files = [obj for obj in response.get('Contents', []) if obj['Size'] > 0]

print(f"Files in s3://{bucket_name}/{prefix}:")
print(f"{'─' * 70}")
for obj in sorted(files, key=lambda x: x['Size'], reverse=True):
    size = obj['Size']
    name = obj['Key'].split('/')[-1]
    if size > 1024 * 1024:
        print(f"  {name:45s} {size / (1024*1024):>10.1f} MB")
    else:
        print(f"  {name:45s} {size:>10,} bytes")

print(f"{'─' * 70}")
total_size = sum(obj['Size'] for obj in files) / (1024 * 1024 * 1024)
print(f"Total files: {len(files)} | Total size: {total_size:.2f} GB")